# NFN — Neural Fractal Network · Google Colab Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AFKmoney/FNN/blob/main/notebooks/colab_train.ipynb)

**Free GPU:** T4 (15 GB) · Sessions up to 12h

### Before you start
1. `Runtime → Change runtime type → T4 GPU`
2. Mount Google Drive (Cell 1) to keep checkpoints between sessions
3. Run all cells — training starts automatically

**Preset:** `small_wikipedia` (~15M params, ~2h on T4, fits in 15GB easily)

In [ ]:
# ── Cell 1: Mount Google Drive (keeps checkpoints between sessions) ───────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/NFN_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✓ Drive mounted → checkpoints will be saved to {DRIVE_DIR}')

In [ ]:
# ── Cell 2: Clone repo & install ─────────────────────────────────────────────
import os

if not os.path.exists('/content/FNN'):
    !git clone https://github.com/AFKmoney/FNN.git /content/FNN
else:
    !git -C /content/FNN pull

os.chdir('/content/FNN')

# Symlink checkpoints to Drive so they persist
DRIVE_DIR = '/content/drive/MyDrive/NFN_checkpoints'
if not os.path.exists('/content/FNN/checkpoints'):
    os.symlink(DRIVE_DIR, '/content/FNN/checkpoints')

!pip install -e . -q
!pip install tqdm -q
print('✓ Ready')

In [ ]:
# ── Cell 3: Verify GPU ────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ {gpu}  ({vram:.1f} GB VRAM)')
else:
    print('⚠ No GPU — go to Runtime → Change runtime type → GPU')

print(f'PyTorch {torch.__version__}')

In [ ]:
# ── Cell 4: Download dataset ──────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/FNN')

from datasets.downloader import DatasetDownloader

dl = DatasetDownloader('/content/FNN/data')
print('Downloading Simple Wikipedia (~120 MB)...')
text = dl.get('wikipedia-en-simple', max_chars=30_000_000)  # 30M chars for speed
print(f'✓ {len(text):,} characters')

os.makedirs('/content/FNN/data', exist_ok=True)
with open('/content/FNN/data/train.txt', 'w') as f:
    f.write(text)
print('✓ Saved')

In [ ]:
# ── Cell 5: Train ─────────────────────────────────────────────────────────────
# small config — 15M params, easily fits T4's 15GB
# Checkpoints auto-save to Google Drive every 500 steps

import os
os.chdir('/content/FNN')

resume_flag = ''
if os.path.exists('checkpoints/agi_nfn_latest.pt'):
    resume_flag = '--resume checkpoints/agi_nfn_latest.pt'
    print('▶ Resuming from Drive checkpoint...')
else:
    print('▶ Starting fresh training...')

!python train_agi.py \
    --text data/train.txt \
    --config small \
    --epochs 5 \
    --batch 8 \
    --seq-len 512 \
    --lr 2e-4 \
    --fp16 \
    --sample-every 200 \
    --save-every 500 \
    {resume_flag}

In [ ]:
# ── Cell 6: Keep-alive (prevents Colab idle timeout) ─────────────────────────
# Run this in a separate browser tab while training
# It clicks the page every 60s to prevent the session from disconnecting

# Paste in browser console (F12 → Console tab):
js_keepalive = """
function keepAlive() {
    var btn = document.querySelector('colab-connect-button');
    if (btn) btn.click();
    setTimeout(keepAlive, 60000);
}
keepAlive();
"""
print('Paste this in your browser console (F12 → Console) to prevent disconnect:')
print(js_keepalive)

In [ ]:
# ── Cell 7: Test generation after training ────────────────────────────────────
import torch, sys
sys.path.insert(0, '/content/FNN')

from nfn.agi_model import build_agi_model
from nfn.tokenizer import NFNTokenizer

tok   = NFNTokenizer()
ckpt  = torch.load('checkpoints/agi_nfn_final.pt', map_location='cpu')
model = build_agi_model(vocab_size=tok.vocab_size, d_model=256, n_blocks=6)
model.load_state_dict(ckpt['model_state'])
model.eval()

for prompt in ["The universe is", "Scientists discovered", "The future of AI"]:
    ids = tok.encode(prompt, add_bos=True)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=80, temperature=0.8)
    print(f'\n[{prompt}]')
    print(tok.decode(out[0].tolist()))

In [ ]:
# ── Cell 8: Resume in new session (run this if Colab restarted) ───────────────
# Run Cell 1 (Drive mount) + Cell 2 (install) first, then this

import os
os.chdir('/content/FNN')

ckpt_path = 'checkpoints/agi_nfn_latest.pt'
if os.path.exists(ckpt_path):
    print(f'✓ Found checkpoint, resuming...')
    !python train_agi.py \
        --text data/train.txt \
        --config small \
        --batch 8 \
        --seq-len 512 \
        --fp16 \
        --epochs 5 \
        --resume {ckpt_path}
else:
    print('No checkpoint on Drive yet — run Cell 4 (download) then Cell 5 (train) first')